# Spare Parts Monthly Sales Analysis

This notebook lists the number of sales per month for every detected spare part in the `revised_sales_2.csv` file and provides statistics on demand and no demand months for each spare part.

## 1. Import Required Libraries
Import pandas and any other necessary libraries for data analysis.

In [4]:
import pandas as pd
import numpy as np


## 2. Load the revised_sales_2.csv File
Read the `revised_sales_2.csv` file into a pandas DataFrame and display the first few rows to inspect the data.

In [9]:
# Load the revised_sales_2.csv file
sales_df = pd.read_csv('../data/revised_sales_2.csv')
sales_df.head()

,reportDate,productName,category,quantity,unitPrice,otherExpenses,totalAmount,customerName,paymentMethod,status,orderNumber,notes
0,2021-01-01,Air Filter,Engine system,2,324.01,56.60,648.02,Retail Client,Cash,Completed,ORD005203,Automated migration
1,2021-01-01,Air Filter,Engine system,3,335.80,82.55,1007.40,Retail Client,Bank Transfer,Completed,ORD005220,Automated migration
2,2021-01-01,Alternator,Electrical system,0,12184.13,0.00,0.00,Wholesale Client,GCash,Completed,ORD022298,Automated migration
3,2021-01-01,Battery,Electrical system,0,5294.91,0.00,0.00,Wholesale Client,Bank Transfer,Completed,ORD021257,Automated migration
4,2021-01-01,Brake Pads,Braking system,1,1381.23,76.66,1381.23,Wholesale Client,Cash,Completed,ORD008852,Automated migration


## 3. List Number of Sales per Month for Each Spare Part
Group the data by spare part and month, then count the number of sales for each combination. Present the results in a pivot table or similar format.

In [ ]:
# Ensure reportDate column is in datetime format and extract year-month
sales_df['reportDate'] = pd.to_datetime(sales_df['reportDate'])
sales_df['year_month'] = sales_df['reportDate'].dt.to_period('M')

# Group by productName and year_month, count sales (quantity > 0)
sales_per_month = sales_df[sales_df['quantity'] > 0].groupby(['productName', 'year_month']).size().reset_index(name='sales_count')

# Pivot table: rows=productName, columns=year_month, values=sales_count
sales_pivot = sales_per_month.pivot(index='productName', columns='year_month', values='sales_count').fillna(0).astype(int)
sales_pivot.head()

KeyError: 'date'

## 4. Calculate Demand and No Demand Months for Each Spare Part
For each spare part, count the number of months with sales (demand months) and months with zero sales (no demand months).

In [ ]:
# Calculate demand and no demand months for each spare part
# All months in the dataset
all_months = sales_pivot.columns

def count_demand_stats(row):
    demand_months = (row > 0).sum()
    no_demand_months = (row == 0).sum()
    return pd.Series({'demand_months': demand_months, 'no_demand_months': no_demand_months})

demand_stats = sales_pivot.apply(count_demand_stats, axis=1)
demand_stats.head()

## 5. Display Statistics for Each Spare Part
Display a summary table showing, for each spare part, the total number of demand months and no demand months.

In [ ]:
# Display summary statistics for each spare part
demand_stats.reset_index(inplace=True)
demand_stats.rename(columns={'spare_part': 'Spare Part'}, inplace=True)
demand_stats.head()

## 6. Business Sales Revenue Statistics per Month
Analyze business sales revenue per month, list the values, and count demand and no demand months where business sales revenue occurred.

In [ ]:
# Group by year_month and sum the business sales revenue
# Replace 'business_sales_revenue' with the actual column name if different
if 'business_sales_revenue' in sales_df.columns:
    revenue_per_month = sales_df.groupby('year_month')['business_sales_revenue'].sum()
else:
    revenue_col = [col for col in sales_df.columns if 'revenue' in col.lower()][0]
    revenue_per_month = sales_df.groupby('year_month')[revenue_col].sum()

revenue_per_month = revenue_per_month.sort_index()
revenue_per_month

In [ ]:
# Count demand and no demand months for business sales revenue
# Demand month: revenue > 0, No demand: revenue == 0
revenue_demand_months = (revenue_per_month > 0).sum()
revenue_no_demand_months = (revenue_per_month == 0).sum()

print(f"Demand months (revenue > 0): {revenue_demand_months}")
print(f"No demand months (revenue == 0): {revenue_no_demand_months}")